# GGSS-R unperturbed-gradient reproduction

This notebook reproduces the fixed-guidance, unperturbed-gradient GGSS-R reconstruction pipeline using the authors’ public code and the FFHQ diffusion checkpoint referenced by the DPS repository. It retains the victim-model preparation, target construction, repository compatibility patches, five pre-reconstruction diagnostics, the full reconstruction run, and the post-reconstruction gradient-identifiability analysis used during development.

The notebook is designed to run from a fresh GPU-backed notebook environment without Google Drive or other platform-specific storage APIs. All persistent artifacts are written beneath a project directory in the current working directory. A CUDA-capable GPU is required for the full experiment.


## 1. Project layout

All paths are defined relative to the directory from which the notebook is launched. This keeps the experiment portable across Colab, local Jupyter, and other hosted notebook environments. The repository, checkpoints, dataset artifacts, and reconstruction outputs are kept in separate subdirectories.


In [1]:
from pathlib import Path
import os, sys

PROJECT_ROOT = (Path.cwd() / "GGSS_R_unperturbed_reproduction").resolve()
REPO_ROOT = PROJECT_ROOT / "repo"
RESULTS_ROOT = PROJECT_ROOT / "results"
MODEL_STATE_ROOT = PROJECT_ROOT / "model_state"
DATA_ROOT = PROJECT_ROOT / "data"
TARGET_PATH = DATA_ROOT / "samples" / "celeba_target" / "target_0000.png"
TARGET_METADATA_PATH = DATA_ROOT / "target_metadata.pt"

for path in [REPO_ROOT, RESULTS_ROOT, MODEL_STATE_ROOT, DATA_ROOT, TARGET_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

CODE_ROOT = str(REPO_ROOT)
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

print("Project root :", PROJECT_ROOT)
print("Repository   :", REPO_ROOT)
print("Results      :", RESULTS_ROOT)
print("Model state  :", MODEL_STATE_ROOT)
print("Data         :", DATA_ROOT)


Project root : /content/GGSS_R_unperturbed_reproduction
Repository   : /content/GGSS_R_unperturbed_reproduction/repo
Results      : /content/GGSS_R_unperturbed_reproduction/results
Model state  : /content/GGSS_R_unperturbed_reproduction/model_state
Data         : /content/GGSS_R_unperturbed_reproduction/data


## 2. Retrieve the authors’ GGSS-R implementation

The public GGSS-R repository stores the runnable source inside `code.zip`. This cell clones the repository only when necessary, extracts that source into the repository root, and records the upstream Git commit so the exact starting point of the experiment remains traceable.


In [2]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path

DRIVE_ROOT = PROJECT_ROOT

REPO_ROOT = (DRIVE_ROOT / "repo").resolve()

REPO_URL = "https://github.com/mjyyhhxx/GGSS-R.git"

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

runner_path = REPO_ROOT / "sample_condition_same_inputs.py"

if runner_path.is_file():

    print("✓ Repository already present on Drive")
    print("  ", REPO_ROOT)

else:

    if REPO_ROOT.exists():
        print("Removing incomplete previous repository...")
        shutil.rmtree(REPO_ROOT)

    print("Cloning GGSS-R repository onto Drive...")

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            REPO_URL,
            str(REPO_ROOT),
        ],
        check=True,
    )

    code_zip = REPO_ROOT / "code.zip"

    assert code_zip.is_file(), (
        f"code.zip not found at {code_zip}"
    )

    print("Extracting code.zip...")

    with zipfile.ZipFile(code_zip, "r") as z:
        z.extractall(REPO_ROOT)

    extracted_code = REPO_ROOT / "code"

    if extracted_code.is_dir():

        for item in extracted_code.iterdir():

            destination = REPO_ROOT / item.name

            if destination.exists():

                if destination.is_dir():
                    shutil.rmtree(destination)
                else:
                    destination.unlink()

            shutil.move(
                str(item),
                str(destination),
            )

        extracted_code.rmdir()

    assert (
        REPO_ROOT /
        "sample_condition_same_inputs.py"
    ).is_file(), (
        "GGSS-R runner was not found after extraction."
    )

    print("✓ Repository cloned and extracted")

CODE_ROOT = str(REPO_ROOT)

os.chdir(CODE_ROOT)

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

print()
print("CODE_ROOT   :", CODE_ROOT)
print("Working dir :", os.getcwd())

try:

    commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        cwd=CODE_ROOT,
        text=True,
    ).strip()

    (DRIVE_ROOT / "AUTHOR_REPO_COMMIT.txt").write_text(
        commit + "\n"
    )

    print("Author repo commit:", commit)

except Exception as exc:

    print(
        "⚠ Could not record Git commit:",
        repr(exc),
    )

print()
print("Top-level files:")

for name in sorted(os.listdir(CODE_ROOT))[:30]:
    print("  ", name)

print()
print("✓ Cell 1 complete")


Removing incomplete previous repository...
Cloning GGSS-R repository onto Drive...
Extracting code.zip...
✓ Repository cloned and extracted

CODE_ROOT   : /content/GGSS_R_unperturbed_reproduction/repo
Working dir : /content/GGSS_R_unperturbed_reproduction/repo
Author repo commit: b337ef1192f3a04a776042882a2d636724817775

Top-level files:
   .DS_Store
   .git
   README.md
   RV
   __MACOSX
   code.zip
   configs
   data
   guided_diffusion
   models
   results
   sample_condition_same_inputs.py
   util
   x_0000.png

✓ Cell 1 complete


## 3. Verify repository structure

Before modifying or executing the authors’ code, the notebook verifies that the runner, diffusion implementation, attacked model, measurement operator, and required configuration files are present. Failing here is preferable to silently running a partial or differently structured checkout.


In [3]:
import os
import sys
from pathlib import Path

assert "CODE_ROOT" in dir() or "CODE_ROOT" in globals(), \
    "CODE_ROOT is not defined. Run Cell 1 first."

CODE_ROOT = str(Path(CODE_ROOT).resolve())
os.chdir(CODE_ROOT)

assert os.path.isdir(CODE_ROOT), f"CODE_ROOT does not exist: {CODE_ROOT}"
assert os.getcwd() == CODE_ROOT

required = [
    "sample_condition_same_inputs.py",
    "guided_diffusion/condition_methods.py",
    "guided_diffusion/measurements.py",
    "guided_diffusion/gaussian_diffusion.py",
    "guided_diffusion/attacked_model.py",
    "configs/model_config.yaml",
    "configs/diffusion_ddim1000_config.yaml",
    "configs/reconstruction_config.yaml",
]

print("Working directory:", os.getcwd())
print()

for relpath in required:
    path = os.path.join(CODE_ROOT, relpath)
    status = os.path.isfile(path)
    print(f"{'✓' if status else '✗'} {relpath}")
    assert status, f"Missing repository file: {path}"

if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

print()
print("✓ Repository integrity check passed.")


Working directory: /content/GGSS_R_unperturbed_reproduction/repo

✓ sample_condition_same_inputs.py
✓ guided_diffusion/condition_methods.py
✓ guided_diffusion/measurements.py
✓ guided_diffusion/gaussian_diffusion.py
✓ guided_diffusion/attacked_model.py
✓ configs/model_config.yaml
✓ configs/diffusion_ddim1000_config.yaml
✓ configs/reconstruction_config.yaml

✓ Repository integrity check passed.


## 4. Install Python dependencies and require CUDA

The experiment depends on LPIPS, Hugging Face Datasets, Opacus, and `gdown` in addition to the libraries normally present in a PyTorch notebook environment. The full 1000-step diffusion reconstruction is GPU-oriented, so this notebook stops immediately if CUDA is unavailable.


In [4]:
import subprocess, sys, torch
packages = ["lpips", "datasets", "opacus", "gdown", "huggingface_hub"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

assert torch.cuda.is_available(), "A CUDA-capable GPU is required for this notebook."
DEVICE = torch.device("cuda:0")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA: 12.8
GPU: Tesla T4


## 5. Download the DPS FFHQ diffusion checkpoint

GGSS-R uses the FFHQ diffusion model distributed through the official Diffusion Posterior Sampling project. The DPS repository links a public Google Drive folder containing `ffhq_10m.pt`. The cell below retrieves that folder only when the required checkpoint is absent, selects `ffhq_10m.pt`, verifies that PyTorch can load it, and installs it at the path expected by GGSS-R.

The download originates from the checkpoint location referenced by the DPS authors rather than from a generic model hub. Because the public Drive link exposes a folder rather than a stable file URL, `gdown` is used to resolve its contents programmatically.


In [5]:
import hashlib, shutil, torch
from pathlib import Path
import gdown

DPS_FOLDER_URL = "https://drive.google.com/drive/folders/1jElnRoFv7b31fG0v6pTSQkelbSX3xGZh"
DPS_DOWNLOAD_DIR = PROJECT_ROOT / "dps-checkpoint"
MODEL_DIR = REPO_ROOT / "models"
MODEL_PATH = MODEL_DIR / "ffhq_10m.pt"
DPS_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

source = DPS_DOWNLOAD_DIR / "ffhq_10m.pt"
if not source.is_file():
    print("Inspecting the public DPS checkpoint folder...")
    manifest = gdown.download_folder(
        url=DPS_FOLDER_URL,
        output=str(DPS_DOWNLOAD_DIR),
        quiet=True,
        use_cookies=False,
        remaining_ok=True,
        skip_download=True,
    )
    ffhq_entries = [entry for entry in manifest if Path(entry.path).name == "ffhq_10m.pt"]
    if len(ffhq_entries) != 1:
        raise RuntimeError(f"Expected exactly one ffhq_10m.pt entry in the DPS folder; found {len(ffhq_entries)}.")
    file_id = ffhq_entries[0].id
    print("Downloading ffhq_10m.pt only...")
    gdown.download(id=file_id, output=str(source), quiet=False, use_cookies=False)

sha = hashlib.sha256()
with source.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(chunk)
print("Checkpoint:", source)
print("Size (MiB):", source.stat().st_size / 1024**2)
print("SHA256:", sha.hexdigest())
_ = torch.load(source, map_location="cpu")
if source.resolve() != MODEL_PATH.resolve():
    shutil.copy2(source, MODEL_PATH)
print("Installed at:", MODEL_PATH)


Inspecting the public DPS checkpoint folder...


Downloading...
From (original): https://drive.google.com/uc?id=1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh
From (redirected): https://drive.google.com/uc?id=1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh&confirm=t&uuid=88964c44-ac6f-471b-ad28-9744bc1b9d2b
To: /content/GGSS_R_unperturbed_reproduction/dps-checkpoint/ffhq_10m.pt
100%|██████████| 374M/374M [00:08<00:00, 46.0MB/s]


Checkpoint: /content/GGSS_R_unperturbed_reproduction/dps-checkpoint/ffhq_10m.pt
Size (MiB): 357.0726709365845
SHA256: 81d535743156ec6be34d8668e6920da94f0614074d7793a16c8fa9e306237faa
Installed at: /content/GGSS_R_unperturbed_reproduction/repo/models/ffhq_10m.pt


## 6. Verify the attacked model and attack class

The reconstruction attack compares gradients from the repository’s `MLP_1` victim model. This cell instantiates the exact architecture and confirms the class convention used by the current GGSS-R measurement path before dataset preparation or training begins.


In [6]:
import sys
import torch
import torch.nn as nn

assert "CODE_ROOT" in globals(), "CODE_ROOT is undefined — run Cell 1 first"
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

from guided_diffusion.attacked_model import MLP_1

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = MLP_1().to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
print("Model:", type(model).__name__)
print(model)
print(f"Trainable parameters: {num_params:,}")

dummy = torch.randn(1, 3, 256, 256, device=DEVICE)
with torch.no_grad():
    output = model(dummy)

print()
print("Input shape :", tuple(dummy.shape))
print("Output shape:", tuple(output.shape))
assert output.shape == (1, 2)

attack_label = torch.tensor([0], dtype=torch.long, device=DEVICE)
criterion = nn.CrossEntropyLoss()
loss = criterion(output, attack_label)

print()
print("Attack label (class index):", attack_label.item())
print("Test CE loss:", float(loss))
print()
print("✓ MLP-1 verified")
print("✓ Attack target class = 0")


Model: MLP_1
MLP_1(
  (fc1): Linear(in_features=196608, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=2, bias=True)
  (relu): ReLU()
)
Trainable parameters: 100,729,730

Input shape : (1, 3, 256, 256)
Output shape: (1, 2)

Attack label (class index): 0
Test CE loss: 0.7272324562072754

✓ MLP-1 verified
✓ Attack target class = 0


## 7. Build the CelebA target artifact

The victim is trained on the CelebA `Smiling` attribute using the same image preprocessing as the existing experiment. This cell loads the training split, selects the class-0 target according to the established experiment logic, and stores both the target image and its metadata under the project directory so later stages use one immutable target artifact.


In [7]:
import random
from pathlib import Path

import torch
from torchvision import transforms
from datasets import load_dataset

DRIVE_ROOT = PROJECT_ROOT

DATA_ROOT = DRIVE_ROOT / "data"

TARGET_DIR = (
    DATA_ROOT /
    "samples" /
    "celeba_target"
)

TARGET_PATH = (
    TARGET_DIR /
    "target_0000.png"
)

TARGET_METADATA_PATH = (
    DATA_ROOT /
    "target_metadata.pt"
)

TARGET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SEED = 2026

random.seed(SEED)
torch.manual_seed(SEED)

CELEBA_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5),
    ),
])

print("=" * 70)
print("LOADING CELEBA TRAINING SPLIT")
print("=" * 70)

celeba = load_dataset(
    "flwrlabs/celeba",
    split="train",
)

print()
print("Training images :", len(celeba))
print("Columns         :", celeba.column_names)

assert "Smiling" in celeba.column_names
assert "image" in celeba.column_names

print()
print("Reading Smiling labels only...")

smiling_labels = celeba["Smiling"]

assert len(smiling_labels) == len(celeba)

class0_indices = [
    i
    for i, label in enumerate(smiling_labels)
    if int(label) == 0
]

class1_indices = [
    i
    for i, label in enumerate(smiling_labels)
    if int(label) == 1
]

print()
print("Victim training population:")
print("  Class 0:", len(class0_indices))
print("  Class 1:", len(class1_indices))

assert len(class0_indices) > 0
assert len(class1_indices) > 0

target_index = random.choice(
    class0_indices
)

print()
print("Selected target index:", target_index)

target_item = celeba[target_index]

target_label = int(
    target_item["Smiling"]
)

assert target_label == 0

target_pil = target_item["image"].convert("RGB")

target_pil.save(
    TARGET_PATH
)

torch.save(
    {
        "target_index": target_index,
        "target_label": target_label,
        "dataset_split": "train",
        "dataset_name": "flwrlabs/celeba",
        "seed": SEED,
    },
    TARGET_METADATA_PATH,
)

assert TARGET_PATH.is_file()
assert TARGET_METADATA_PATH.is_file()

print()
print("=" * 70)
print("TARGET SELECTION")
print("=" * 70)

print("Target index :", target_index)
print("Target label :", target_label)
print("Target split :", "train")
print("Target path  :", TARGET_PATH)
print("Image size   :", target_pil.size)

print()
print("✓ Victim training population contains class 0")
print("✓ Victim training population contains class 1")
print("✓ Target is selected ONLY from the training split")
print("✓ Target label is strictly 0")
print("✓ Only the selected target image was decoded")
print()
print("✓ Cell 6 complete")


LOADING CELEBA TRAINING SPLIT


README.md:   0%|          | 0.00/9.28k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

img_align+identity+attr/train-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  500MB            

...

Generating train split:   0%|          | 0/162770 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/19867 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19962 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]


Training images : 162770
Columns         : ['image', 'celeb_id', '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']

Reading Smiling labels only...

Victim training population:
  Class 0: 84690
  Class 1: 78080

Selected target index: 29786

TARGET SELECTION
Target index : 29786
Target label : 0
Target split : train
Target path  : /content/GGSS_R_unperturbed_reproduction/data/samples/celeba_target/target_0000.png
Image size   : (178, 218)

✓ Victim training popu

## 8. Train or restore the victim model

The GGSS-R measurement is the gradient of the victim loss, so reconstruction depends directly on the victim checkpoint. This cell preserves the existing full-training procedure for `MLP_1`, including its optimizer, preprocessing, batch size, and checkpoint-resume behavior, while storing the resulting model beneath the portable project root.


In [8]:
import os
import random
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

SEED = 2026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DRIVE_ROOT = PROJECT_ROOT

DRIVE_CKPT_DIR = (
    DRIVE_ROOT /
    "model_state"
)

DRIVE_CKPT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LATEST_CKPT = (
    DRIVE_CKPT_DIR /
    "MLP_1.pth"
)

REPO_CKPT_DIR = (
    Path(CODE_ROOT) /
    "model_state"
)

REPO_CKPT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPO_CKPT = (
    REPO_CKPT_DIR /
    "MLP_1.pth"
)

DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

VICTIM_TRAIN_BATCH_SIZE = 64

NUM_WORKERS = 2

EPOCHS = 1

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-5

print("=" * 70)
print("VICTIM MODEL TRAINING")
print("=" * 70)

print("Device                    :", DEVICE)
print("Victim training batch     :", VICTIM_TRAIN_BATCH_SIZE)
print("Requested epochs          :", EPOCHS)
print("Learning rate             :", LEARNING_RATE)
print("Weight decay              :", WEIGHT_DECAY)
print("Drive checkpoint dir      :", DRIVE_CKPT_DIR)
print("Repository checkpoint    :", REPO_CKPT)

class CelebAMLPDataset(Dataset):

    def __init__(
        self,
        hf_dataset,
        transform,
    ):

        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):

        return len(self.dataset)

    def __getitem__(
        self,
        index,
    ):

        item = self.dataset[index]

        image = self.transform(
            item["image"]
        )

        label = torch.tensor(
            int(item["Smiling"]),
            dtype=torch.long,
        )

        return image, label

assert "celeba" in globals()
assert celeba is not None

train_dataset = CelebAMLPDataset(
    celeba,
    CELEBA_TRANSFORM,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=VICTIM_TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print()
print("Training samples :", len(train_dataset))
print("Training batches :", len(train_loader))

def find_epoch_checkpoints():

    checkpoints = []

    for path in DRIVE_CKPT_DIR.glob(
        "MLP_1_epoch_*.pth"
    ):

        try:

            epoch_number = int(
                path.stem.split("_")[-1]
            )

            checkpoints.append(
                (epoch_number, path)
            )

        except ValueError:
            pass

    checkpoints.sort(
        key=lambda x: x[0]
    )

    return checkpoints

completed = find_epoch_checkpoints()

if completed:

    latest_epoch, latest_path = completed[-1]

    print()
    print(
        f"✓ Found latest completed victim "
        f"checkpoint: epoch {latest_epoch}"
    )
    print("  ", latest_path)

    shutil.copy2(
        latest_path,
        LATEST_CKPT,
    )

    shutil.copy2(
        latest_path,
        REPO_CKPT,
    )

    print()
    print("✓ Latest completed checkpoint selected")
    print("  Epoch :", latest_epoch)
    print("  Drive :", LATEST_CKPT)
    print("  Repo  :", REPO_CKPT)

else:

    from guided_diffusion.attacked_model import MLP_1

    model = MLP_1().to(DEVICE)

    num_params = sum(
        p.numel()
        for p in model.parameters()
    )

    print()
    print(
        f"Model parameters: {num_params:,}"
    )

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    model.train()

    try:

        for epoch in range(EPOCHS):

            running_loss = 0.0
            correct = 0
            total = 0

            print()
            print(
                f"Epoch {epoch + 1}/{EPOCHS}"
            )

            for batch_idx, (
                images,
                labels,
            ) in enumerate(train_loader):

                images = images.to(
                    DEVICE,
                    non_blocking=True,
                )

                labels = labels.to(
                    DEVICE,
                    non_blocking=True,
                )

                optimizer.zero_grad(
                    set_to_none=True
                )

                outputs = model(
                    images
                )

                loss = criterion(
                    outputs,
                    labels,
                )

                if not torch.isfinite(loss):

                    raise RuntimeError(
                        "Non-finite training loss "
                        f"at epoch {epoch + 1}, "
                        f"batch {batch_idx}: "
                        f"{loss.item()}"
                    )

                loss.backward()

                optimizer.step()

                bs = labels.size(0)

                running_loss += (
                    loss.item() * bs
                )

                correct += (
                    outputs.argmax(dim=1)
                    == labels
                ).sum().item()

                total += bs

                if (
                    batch_idx == 0
                    or
                    (batch_idx + 1) % 250 == 0
                    or
                    (batch_idx + 1)
                    == len(train_loader)
                ):

                    print(
                        f"  batch "
                        f"{batch_idx + 1:5d}/"
                        f"{len(train_loader):5d} | "
                        f"loss "
                        f"{running_loss / total:.4f} | "
                        f"acc "
                        f"{100.0 * correct / total:.2f}%"
                    )

            epoch_ckpt = (
                DRIVE_CKPT_DIR /
                f"MLP_1_epoch_{epoch + 1:02d}.pth"
            )

            torch.save(
                model.state_dict(),
                epoch_ckpt,
            )

            torch.save(
                model.state_dict(),
                LATEST_CKPT,
            )

            torch.save(
                model.state_dict(),
                REPO_CKPT,
            )

            print()
            print(
                f"✓ Completed epoch {epoch + 1}"
            )

            print(
                "  checkpoint:",
                epoch_ckpt,
            )

    except Exception as exc:

        print()
        print("=" * 70)
        print("TRAINING INTERRUPTED")
        print("=" * 70)

        completed_after_failure = (
            find_epoch_checkpoints()
        )

        if completed_after_failure:

            latest_epoch, latest_path = (
                completed_after_failure[-1]
            )

            print(
                f"Latest fully completed epoch: "
                f"{latest_epoch}"
            )

            print(
                "Checkpoint:",
                latest_path,
            )

            shutil.copy2(
                latest_path,
                LATEST_CKPT,
            )

            shutil.copy2(
                latest_path,
                REPO_CKPT,
            )

            print()
            print(
                "✓ Latest completed checkpoint "
                "has been selected."
            )

        else:

            print(
                "✗ No completed epoch checkpoint exists."
            )

        raise

assert LATEST_CKPT.is_file()
assert REPO_CKPT.is_file()

print()
print("=" * 70)
print("VICTIM MODEL READY")
print("=" * 70)

print("Latest checkpoint:", LATEST_CKPT)
print("Repository copy  :", REPO_CKPT)

print()
print("✓ Victim trained on FULL CelebA training split")
print("✓ Victim training batch size =", VICTIM_TRAIN_BATCH_SIZE)
print("✓ Latest completed epoch checkpoint selected")
print("✓ Cell 7 complete")


VICTIM MODEL TRAINING
Device                    : cuda:0
Victim training batch     : 64
Requested epochs          : 1
Learning rate             : 0.0001
Weight decay              : 1e-05
Drive checkpoint dir      : /content/GGSS_R_unperturbed_reproduction/model_state
Repository checkpoint    : /content/GGSS_R_unperturbed_reproduction/repo/model_state/MLP_1.pth

Training samples : 162770
Training batches : 2543

Model parameters: 100,729,730

Epoch 1/1
  batch     1/ 2543 | loss 0.7042 | acc 39.06%
  batch   250/ 2543 | loss 0.4531 | acc 80.40%
  batch   500/ 2543 | loss 0.4025 | acc 82.68%
  batch   750/ 2543 | loss 0.3713 | acc 84.06%
  batch  1000/ 2543 | loss 0.3557 | acc 84.79%
  batch  1250/ 2543 | loss 0.3450 | acc 85.31%
  batch  1500/ 2543 | loss 0.3343 | acc 85.76%
  batch  1750/ 2543 | loss 0.3253 | acc 86.15%
  batch  2000/ 2543 | loss 0.3188 | acc 86.47%
  batch  2250/ 2543 | loss 0.3132 | acc 86.75%
  batch  2500/ 2543 | loss 0.3089 | acc 86.94%
  batch  2543/ 2543 | loss 

## 9. Verify the completed victim checkpoint

Before the attack is run, the latest victim checkpoint is validated and copied into the repository location expected by the original measurement operator. This prevents the attack from accidentally using a stale or missing model.


In [9]:
from pathlib import Path
import shutil
import torch

assert "CODE_ROOT" in globals()

DRIVE_ROOT = PROJECT_ROOT

DRIVE_CKPT = (
    DRIVE_ROOT /
    "model_state" /
    "MLP_1.pth"
)

REPO_CKPT = (
    Path(CODE_ROOT) /
    "model_state" /
    "MLP_1.pth"
)

assert DRIVE_CKPT.is_file(), (
    f"Missing victim checkpoint:\n{DRIVE_CKPT}"
)

REPO_CKPT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if (
    not REPO_CKPT.is_file()
    or
    REPO_CKPT.stat().st_size
    != DRIVE_CKPT.stat().st_size
):

    shutil.copy2(
        DRIVE_CKPT,
        REPO_CKPT,
    )

    print(
        "✓ Copied latest victim checkpoint "
        "into repository."
    )

else:

    print(
        "✓ Repository checkpoint already matches "
        "the Drive checkpoint."
    )

CHECKPOINT_PATH = str(
    REPO_CKPT
)

CHECKPOINT_DIR = str(
    REPO_CKPT.parent
)

print()
print("Drive checkpoint :", DRIVE_CKPT)
print("Repo checkpoint  :", REPO_CKPT)
print(
    "Size             : "
    f"{DRIVE_CKPT.stat().st_size / (1024**2):.2f} MB"
)

print()
print("✓ MLP_1 checkpoint ready")


✓ Repository checkpoint already matches the Drive checkpoint.

Drive checkpoint : /content/GGSS_R_unperturbed_reproduction/model_state/MLP_1.pth
Repo checkpoint  : /content/GGSS_R_unperturbed_reproduction/repo/model_state/MLP_1.pth
Size             : 384.26 MB

✓ MLP_1 checkpoint ready


## 10. Victim–target sanity check

This sanity check reloads the saved target with the attack preprocessing and evaluates it through the trained victim. It verifies that the target artifact, target metadata, victim checkpoint, and class convention are mutually consistent before any gradient diagnostics are interpreted.


In [10]:
import torch
from pathlib import Path
from PIL import Image
from torchvision import transforms

assert "DEVICE" in globals()
assert "CHECKPOINT_PATH" in globals()

TARGET_PATH = TARGET_PATH

TARGET_METADATA_PATH = TARGET_METADATA_PATH

assert TARGET_PATH.is_file()
assert TARGET_METADATA_PATH.is_file()

CELEBA_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5),
    ),
])

metadata = torch.load(
    TARGET_METADATA_PATH,
    map_location="cpu",
)

assert metadata["dataset_split"] == "train"
assert metadata["target_label"] == 0

print("=" * 70)
print("TARGET METADATA")
print("=" * 70)

print("Dataset       :", metadata["dataset_name"])
print("Split         :", metadata["dataset_split"])
print("Target index  :", metadata["target_index"])
print("Target label  :", metadata["target_label"])

from guided_diffusion.attacked_model import MLP_1

model = MLP_1().to(DEVICE)

model.load_state_dict(
    torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
    )
)

model.eval()

print()
print("✓ Victim checkpoint loaded")
print("  Device:", DEVICE)

target_pil = Image.open(
    TARGET_PATH
).convert("RGB")

target_tensor = (
    CELEBA_TRANSFORM(target_pil)
    .unsqueeze(0)
    .to(DEVICE)
)

with torch.no_grad():

    logits = model(
        target_tensor
    )

    probabilities = torch.softmax(
        logits,
        dim=1,
    )[0]

    prediction = logits.argmax(
        dim=1
    ).item()

    confidence = probabilities[
        prediction
    ].item()

print()
print("=" * 70)
print("TARGET VICTIM-MODEL CHECK")
print("=" * 70)

print(
    "Expected label :", 0
)

print(
    "Predicted label:",
    prediction,
)

print(
    f"Class-0 prob   : {probabilities[0].item():.6f}"
)

print(
    f"Class-1 prob   : {probabilities[1].item():.6f}"
)

print(
    f"Prediction confidence: {confidence:.6f}"
)

assert prediction == 0, (
    "The selected class-0 training image is not "
    "currently predicted as class 0 by the victim. "
    "Do not launch GGSS-R yet."
)

print()
print("✓ Target metadata says label = 0")
print("✓ Victim predicts target as class 0")
print("✓ Target is suitable for the class-0 GGSS-R measurement")


TARGET METADATA
Dataset       : flwrlabs/celeba
Split         : train
Target index  : 29786
Target label  : 0

✓ Victim checkpoint loaded
  Device: cuda:0

TARGET VICTIM-MODEL CHECK
Expected label : 0
Predicted label: 0
Class-0 prob   : 0.986194
Class-1 prob   : 0.013806
Prediction confidence: 0.986194

✓ Target metadata says label = 0
✓ Victim predicts target as class 0
✓ Target is suitable for the class-0 GGSS-R measurement


## 11. Final artifact verification

The reconstruction runner requires the model configuration, 1000-step DDIM configuration, reconstruction task configuration, diffusion checkpoint, victim checkpoint, and target image. This cell checks all of those artifacts together before any source patches or diagnostics are executed.


In [11]:
from pathlib import Path

assert "CODE_ROOT" in globals()

RECON_CONFIG = (
    Path(CODE_ROOT) /
    "configs" /
    "reconstruction_config.yaml"
)

DIFFUSION_CONFIG = (
    Path(CODE_ROOT) /
    "configs" /
    "diffusion_ddim1000_config.yaml"
)

RUNNER = (
    Path(CODE_ROOT) /
    "sample_condition_same_inputs.py"
)

MLP_CHECKPOINT = (
    Path(CODE_ROOT) /
    "model_state" /
    "MLP_1.pth"
)

DIFFUSION_CHECKPOINT = (
    Path(CODE_ROOT) /
    "models" /
    "ffhq_10m.pt"
)

TARGET = TARGET_PATH

TARGET_METADATA = TARGET_METADATA_PATH

required_files = {
    "reconstruction config"     : RECON_CONFIG,
    "diffusion config"          : DIFFUSION_CONFIG,
    "GGSS-R runner"             : RUNNER,
    "MLP_1 checkpoint"           : MLP_CHECKPOINT,
    "DPS FFHQ checkpoint"        : DIFFUSION_CHECKPOINT,
    "CelebA target"              : TARGET,
    "target metadata"            : TARGET_METADATA,
}

print("=" * 70)
print("GGSS-R ARTIFACT CHECK")
print("=" * 70)

all_ok = True

for name, path in required_files.items():

    exists = path.is_file()

    print(
        f"{'✓' if exists else '✗'} "
        f"{name}: {path}"
    )

    if not exists:
        all_ok = False

print()

if not all_ok:

    raise FileNotFoundError(
        "One or more required artifacts are missing."
    )

print("=" * 70)
print("✓ ALL REQUIRED ARTIFACTS PRESENT")
print("=" * 70)

print()
print("Ready for gradient preflight.")


GGSS-R ARTIFACT CHECK
✓ reconstruction config: /content/GGSS_R_unperturbed_reproduction/repo/configs/reconstruction_config.yaml
✓ diffusion config: /content/GGSS_R_unperturbed_reproduction/repo/configs/diffusion_ddim1000_config.yaml
✓ GGSS-R runner: /content/GGSS_R_unperturbed_reproduction/repo/sample_condition_same_inputs.py
✓ MLP_1 checkpoint: /content/GGSS_R_unperturbed_reproduction/repo/model_state/MLP_1.pth
✓ DPS FFHQ checkpoint: /content/GGSS_R_unperturbed_reproduction/repo/models/ffhq_10m.pt
✓ CelebA target: /content/GGSS_R_unperturbed_reproduction/data/samples/celeba_target/target_0000.png
✓ target metadata: /content/GGSS_R_unperturbed_reproduction/data/target_metadata.pt

✓ ALL REQUIRED ARTIFACTS PRESENT

Ready for gradient preflight.


## 12. Patch reconstruction progress-directory creation

The upstream runner writes progress images to a directory whose creation is inconsistent with the write path. This minimal compatibility patch creates the directory before writing so the original reconstruction logic can complete without a filesystem error.


In [12]:
from pathlib import Path
import shutil

assert "CODE_ROOT" in globals(), "Run Cell 1 first"

RUNNER_PATH = Path(CODE_ROOT) / "sample_condition_same_inputs.py"
assert RUNNER_PATH.is_file(), f"Runner not found: {RUNNER_PATH}"

backup = RUNNER_PATH.with_suffix(".py.bak")
if not backup.exists():
    shutil.copy2(RUNNER_PATH, backup)
    print("✓ Backup created:", backup.name)

text = RUNNER_PATH.read_text()

old = """        sample, distance_list = sample_fn(x_start=x_start, measurement=y_n, record=True,
                                          save_root=os.path.join(out_path, 'progresss', str(i).zfill(5)))"""

new = """        progress_dir = os.path.join(
            out_path,
            'progresss',
            str(i).zfill(5)
        )
        os.makedirs(progress_dir, exist_ok=True)

        sample, distance_list = sample_fn(
            x_start=x_start,
            measurement=y_n,
            record=True,
            save_root=progress_dir
        )"""

if "os.makedirs(progress_dir, exist_ok=True)" in text:
    print("✓ Patch already present — nothing to do")
else:
    if old not in text:
        raise RuntimeError(
            "Expected runner block not found.\n"
            "The file may have been modified already or differs from the expected version."
        )
    text = text.replace(old, new, 1)
    RUNNER_PATH.write_text(text)
    print("✓ Surgical patch applied")

patched = RUNNER_PATH.read_text()
assert "progress_dir = os.path.join(" in patched
assert "os.makedirs(progress_dir, exist_ok=True)" in patched
assert "save_root=progress_dir" in patched

print("✓ Patch verification passed")
print("Modified file:", RUNNER_PATH)


✓ Backup created: sample_condition_same_inputs.py.bak
✓ Surgical patch applied
✓ Patch verification passed
Modified file: /content/GGSS_R_unperturbed_reproduction/repo/sample_condition_same_inputs.py


## 13. Patch predicted-x0 progress-directory creation

The diffusion implementation also writes predicted clean-image snapshots during sampling. This second minimal patch ensures that the corresponding `x0` directory exists before those snapshots are saved.


In [13]:
from pathlib import Path
import shutil

assert "CODE_ROOT" in globals(), "Run Cell 1 first"

GD_PATH = Path(CODE_ROOT) / "guided_diffusion" / "gaussian_diffusion.py"
assert GD_PATH.is_file(), f"File not found: {GD_PATH}"

backup = GD_PATH.with_suffix(".py.bak")
if not backup.exists():
    shutil.copy2(GD_PATH, backup)
    print("✓ Backup created:", backup.name)

text = GD_PATH.read_text()

old = """                x0_file_path = os.path.join('code/results/reconstruction/progress/x0', f"x_{str(idx).zfill(4)}.png")
                plt.imsave(file_path, clear_color(img))
                plt.imsave(x0_file_path, clear_color(out['pred_xstart']) )"""

new = """                x0_file_path = os.path.join(save_root, 'x0', f"x_{str(idx).zfill(4)}.png")
                os.makedirs(os.path.dirname(x0_file_path), exist_ok=True)
                plt.imsave(file_path, clear_color(img))
                plt.imsave(x0_file_path, clear_color(out['pred_xstart']) )"""

if "os.makedirs(os.path.dirname(x0_file_path), exist_ok=True)" in text:
    print("✓ x0 directory patch already present — nothing to do")
else:
    if old not in text:
        raise RuntimeError(
            "Expected x0 save block not found.\n"
            "The file may already have been modified or differs from the expected version."
        )
    text = text.replace(old, new, 1)
    GD_PATH.write_text(text)
    print("✓ Surgical x0 directory patch applied")

patched = GD_PATH.read_text()
assert "os.makedirs(os.path.dirname(x0_file_path), exist_ok=True)" in patched
assert "plt.imsave(x0_file_path" in patched

print("✓ Patch verification passed")
print("Modified file:", GD_PATH)


✓ Backup created: gaussian_diffusion.py.bak
✓ Surgical x0 directory patch applied
✓ Patch verification passed
Modified file: /content/GGSS_R_unperturbed_reproduction/repo/guided_diffusion/gaussian_diffusion.py


## 14. Save sparse diffusion progress snapshots

Saving every one of 1000 diffusion steps creates unnecessary storage and I/O overhead. This patch preserves the reconstruction itself while saving diagnostic snapshots only every 100 steps and at the final step, which is sufficient for tracing the reconstruction trajectory.


In [14]:
from pathlib import Path
import re
import shutil

assert "CODE_ROOT" in globals(), "Run Cell 1 first"

GD_PATH = Path(CODE_ROOT) / "guided_diffusion" / "gaussian_diffusion.py"
assert GD_PATH.is_file(), f"File not found: {GD_PATH}"

backup = GD_PATH.with_suffix(".py.bak_sparse")
if not backup.exists():
    shutil.copy2(GD_PATH, backup)
    print("✓ Backup created:", backup.name)

text = GD_PATH.read_text()

print("=" * 70)
print("CURRENT SAVE BLOCK (for inspection)")
print("=" * 70)

match = re.search(
    r"(if record:.*?)(?=\n\s*return img, distance_list)",
    text,
    re.DOTALL
)

if match is None:
    raise RuntimeError("Could not locate the 'if record:' block at all.")

current_block = match.group(1)
print(current_block)
print("=" * 70)

new_block = '''if record and (idx % 100 == 0 or idx == 0):
                file_path = os.path.join(save_root, f"x_{str(idx).zfill(4)}.png")
                x0_file_path = os.path.join(save_root, "x0", f"x_{str(idx).zfill(4)}.png")
                os.makedirs(os.path.dirname(x0_file_path), exist_ok=True)
                plt.imsave(file_path, clear_color(img))
                plt.imsave(x0_file_path, clear_color(out['pred_xstart']))
'''

if "if record and (idx % 100 == 0 or idx == 0):" in text:
    print("✓ Sparse-saving patch already present — nothing to do")
else:
    text = text[:match.start(1)] + new_block + text[match.end(1):]
    GD_PATH.write_text(text)
    print("✓ Sparse progress-saving patch applied")

verify = GD_PATH.read_text()
assert "if record and (idx % 100 == 0 or idx == 0):" in verify
assert 'os.path.join(save_root, "x0"' in verify or "os.path.join(save_root, 'x0'" in verify

print()
print("✓ Verification passed")
print("  - Saves every 100 steps")
print("  - Always saves final state (idx == 0)")
print("  - x0 images go under save_root/x0/")
print("Modified file:", GD_PATH)


✓ Backup created: gaussian_diffusion.py.bak_sparse
CURRENT SAVE BLOCK (for inspection)
if record:
                file_path = os.path.join(save_root, f"x_{str(idx).zfill(4)}.png")
                x0_file_path = os.path.join(save_root, 'x0', f"x_{str(idx).zfill(4)}.png")
                os.makedirs(os.path.dirname(x0_file_path), exist_ok=True)
                plt.imsave(file_path, clear_color(img))
                plt.imsave(x0_file_path, clear_color(out['pred_xstart']) )
✓ Sparse progress-saving patch applied

✓ Verification passed
  - Saves every 100 steps
  - Always saves final state (idx == 0)
  - x0 images go under save_root/x0/
Modified file: /content/GGSS_R_unperturbed_reproduction/repo/guided_diffusion/gaussian_diffusion.py


## 15. Verify patched paths before diagnostics

This final pre-diagnostic check confirms that the patched diffusion source and the intended baseline output directory resolve under the portable project tree.


In [15]:
from pathlib import Path

REPO_ROOT = REPO_ROOT

RESULT_ROOT = RESULTS_ROOT

GD_PATH = REPO_ROOT / "guided_diffusion" / "gaussian_diffusion.py"
RUN_DIR = RESULT_ROOT / "baseline_fixed_mr_v3"

print("=" * 70)
print("GGSS-R PATH & PATCH CHECK")
print("=" * 70)
print("Repository :", REPO_ROOT)
print("Results    :", RESULT_ROOT)
print("Run dir    :", RUN_DIR)
print()

assert REPO_ROOT.is_dir()
assert GD_PATH.is_file()
assert RESULT_ROOT.is_dir()

gd_text = GD_PATH.read_text()
assert "if record and (idx % 100 == 0 or idx == 0):" in gd_text, \
    "Sparse-saving patch is missing"
assert 'os.path.join(save_root, "x0"' in gd_text or \
       "os.path.join(save_root, 'x0'" in gd_text, \
    "x0 path under save_root is missing"

print("✓ Drive repository exists")
print("✓ gaussian_diffusion.py exists")
print("✓ Sparse 100-step patch is present")
print("✓ Run directory is:", "absent (clean)" if not RUN_DIR.exists() else "ALREADY EXISTS")
print()
print("✓ PATH & PATCH CHECK PASSED")


GGSS-R PATH & PATCH CHECK
Repository : /content/GGSS_R_unperturbed_reproduction/repo
Results    : /content/GGSS_R_unperturbed_reproduction/results
Run dir    : /content/GGSS_R_unperturbed_reproduction/results/baseline_fixed_mr_v3

✓ Drive repository exists
✓ gaussian_diffusion.py exists
✓ Sparse 100-step patch is present
✓ Run directory is: absent (clean)

✓ PATH & PATCH CHECK PASSED


## Pre-reconstruction diagnostic 1 — Target-gradient validity

GGSS-R can only reconstruct an image if the selected target produces a meaningful victim gradient. This diagnostic isolates the target from the diffusion pipeline and computes the exact parameter gradient used as the attack measurement. It checks its norm, sparsity, and autograd graph so a later reconstruction failure is not mistakenly attributed to an invalid target gradient.


In [16]:
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from torchvision import transforms

assert "DEVICE" in globals()
assert "CHECKPOINT_PATH" in globals()

print("=" * 70)
print("GRADIENT DIAGNOSTIC 1 — TARGET-ONLY GRADIENT")
print("=" * 70)

from guided_diffusion.attacked_model import MLP_1

victim = MLP_1().to(DEVICE)

victim.load_state_dict(
    torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
    )
)

victim.eval()

TARGET_PATH = TARGET_PATH

target_pil = Image.open(
    TARGET_PATH
).convert("RGB")

attack_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5),
    ),
])

target_x = (
    attack_transform(target_pil)
    .unsqueeze(0)
    .to(DEVICE)
)

target_x.requires_grad_(True)

target_y = torch.tensor(
    [0],
    dtype=torch.long,
    device=DEVICE,
)

criterion = nn.CrossEntropyLoss()

victim.zero_grad(
    set_to_none=True
)

target_logits = victim(
    target_x
)

target_loss = criterion(
    target_logits,
    target_y,
)

target_grads = torch.autograd.grad(
    target_loss,
    tuple(victim.parameters()),
    create_graph=True,
    retain_graph=True,
    allow_unused=False,
)

print()
print("Target logits:")
print(
    target_logits.detach().cpu()
)

print()
print(
    f"Target loss: {target_loss.item():.8f}"
)

total_sq_norm = torch.zeros(
    (),
    device=DEVICE,
)

total_nonzero = 0
total_elements = 0

print()
print("-" * 70)
print("PER-LAYER GRADIENTS")
print("-" * 70)

for (name, parameter), gradient in zip(
    victim.named_parameters(),
    target_grads,
):

    grad_norm = gradient.norm()

    nonzero = (
        gradient.detach()
        .ne(0)
        .sum()
        .item()
    )

    numel = gradient.numel()

    total_sq_norm = (
        total_sq_norm
        + gradient.pow(2).sum()
    )

    total_nonzero += nonzero
    total_elements += numel

    print(
        f"{name:35s} "
        f"shape={str(tuple(gradient.shape)):22s} "
        f"norm={grad_norm.item():.8e} "
        f"nonzero={nonzero:,}/{numel:,} "
        f"grad_fn={gradient.grad_fn is not None}"
    )

total_norm = torch.sqrt(
    total_sq_norm
)

print()
print("=" * 70)
print("TARGET GRADIENT SUMMARY")
print("=" * 70)

print(
    f"Total gradient norm       : "
    f"{total_norm.item():.10e}"
)

print(
    f"Nonzero gradient elements : "
    f"{total_nonzero:,} / {total_elements:,}"
)

print(
    f"Nonzero fraction          : "
    f"{total_nonzero / total_elements:.8f}"
)

print(
    "Gradient graph preserved  :",
    all(
        g.grad_fn is not None
        for g in target_grads
    )
)

assert target_loss.requires_grad
assert len(target_grads) == len(
    tuple(victim.parameters())
)

assert all(
    g.numel() > 0
    for g in target_grads
)

assert total_norm.item() > 0, (
    "TARGET GRADIENT NORM IS ZERO. "
    "This is already a critical failure."
)

print()
print("✓ Target gradient is nonzero")
print("✓ Parameter gradients were obtained")
print("✓ Autograd graph was preserved")
print()
print("✓ DIAGNOSTIC 1 COMPLETE")


GRADIENT DIAGNOSTIC 1 — TARGET-ONLY GRADIENT

Target logits:
tensor([[-0.1091, -4.3779]])

Target loss: 0.01390261

----------------------------------------------------------------------
PER-LAYER GRADIENTS
----------------------------------------------------------------------
fc1.weight                          shape=(512, 196608)          norm=1.43323883e-01 nonzero=17,104,896/100,663,296 grad_fn=True
fc1.bias                            shape=(512,)                 norm=6.15877041e-04 nonzero=87/512 grad_fn=True
fc2.weight                          shape=(128, 512)             norm=9.47894633e-01 nonzero=2,349/65,536 grad_fn=True
fc2.bias                            shape=(128,)                 norm=2.95649492e-03 nonzero=27/128 grad_fn=True
fc3.weight                          shape=(2, 128)               norm=9.30052400e-01 nonzero=54/256 grad_fn=True
fc3.bias                            shape=(2,)                   norm=1.95252020e-02 nonzero=2/2 grad_fn=True

TARGET GRADIENT SUMMARY


### Results interpretation



## Pre-reconstruction diagnostic 2 — Gradient validity for an unrelated real image

A nonzero target gradient alone does not show that the victim responds meaningfully to different inputs. This diagnostic sends a separate CelebA image through the same victim and loss path and verifies that an ordinary, unrelated image also produces a valid gradient. The comparison helps distinguish a target-specific anomaly from normal behavior of the victim-gradient measurement.


In [17]:
print("=" * 70)
print("GRADIENT DIAGNOSTIC 2 — UNRELATED IMAGE")
print("=" * 70)

assert "victim" in globals()
assert "DEVICE" in globals()
assert "criterion" in globals()
assert "attack_transform" in globals()

from datasets import load_dataset

print()
print("Loading CelebA train split for unrelated-image check...")

celeba_diag2 = load_dataset(
    "flwrlabs/celeba",
    split="train",
)

unrelated_index = 0

unrelated_item = celeba_diag2[
    unrelated_index
]

unrelated_pil = (
    unrelated_item["image"]
    .convert("RGB")
)

unrelated_label = int(
    unrelated_item["Smiling"]
)

unrelated_x = (
    attack_transform(unrelated_pil)
    .unsqueeze(0)
    .to(DEVICE)
)

unrelated_x.requires_grad_(True)

unrelated_y = torch.tensor(
    [unrelated_label],
    dtype=torch.long,
    device=DEVICE,
)

victim.zero_grad(
    set_to_none=True
)

unrelated_logits = victim(
    unrelated_x
)

unrelated_loss = criterion(
    unrelated_logits,
    unrelated_y
)

unrelated_grads = torch.autograd.grad(
    unrelated_loss,
    tuple(victim.parameters()),
    create_graph=True,
    retain_graph=True,
    allow_unused=False,
)

unrelated_sq_norm = torch.zeros(
    (),
    device=DEVICE,
)

unrelated_nonzero = 0
unrelated_elements = 0

for gradient in unrelated_grads:

    unrelated_sq_norm += (
        gradient.pow(2).sum()
    )

    unrelated_nonzero += (
        gradient.detach()
        .ne(0)
        .sum()
        .item()
    )

    unrelated_elements += (
        gradient.numel()
    )

unrelated_norm = torch.sqrt(
    unrelated_sq_norm
)

print()
print(
    "Unrelated image index :",
    unrelated_index
)

print(
    "Unrelated image label :",
    unrelated_label
)

print()
print(
    "Logits:",
    unrelated_logits.detach().cpu()
)

print(
    f"Loss                    : "
    f"{unrelated_loss.item():.8f}"
)

print(
    f"Gradient norm           : "
    f"{unrelated_norm.item():.10e}"
)

print(
    f"Nonzero gradient values : "
    f"{unrelated_nonzero:,} / "
    f"{unrelated_elements:,}"
)

print(
    f"Nonzero fraction        : "
    f"{unrelated_nonzero / unrelated_elements:.8f}"
)

print(
    "Gradient graph preserved:",
    all(
        g.grad_fn is not None
        for g in unrelated_grads
    )
)

assert unrelated_norm.item() > 0, (
    "UNRELATED IMAGE GRADIENT IS ZERO."
)

print()
print("✓ Ordinary image produces a nonzero gradient")
print("✓ DIAGNOSTIC 2 COMPLETE")


GRADIENT DIAGNOSTIC 2 — UNRELATED IMAGE

Loading CelebA train split for unrelated-image check...


Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]


Unrelated image index : 0
Unrelated image label : 1

Logits: tensor([[-1.5298, -1.1498]])
Loss                    : 0.52109510
Gradient norm           : 3.5276134491e+01
Nonzero gradient values : 22,613,577 / 100,729,730
Nonzero fraction        : 0.22449754
Gradient graph preserved: True

✓ Ordinary image produces a nonzero gradient
✓ DIAGNOSTIC 2 COMPLETE


### Results interpretation



## Pre-reconstruction diagnostic 3 — Differentiability of gradient distance with respect to the image

GGSS-R guides diffusion by differentiating a distance between the leaked target gradient and the gradient induced by a candidate image. The entire attack therefore depends on a second-order autograd path from parameter-gradient distance back to image pixels. This diagnostic constructs that path directly and verifies that the resulting image gradient is finite and nonzero.


In [18]:
print("=" * 70)
print("GRADIENT DIAGNOSTIC 3 — SECOND-ORDER DIFFERENTIABILITY")
print("=" * 70)

x = unrelated_x.detach().clone()

x.requires_grad_(True)

victim.zero_grad(
    set_to_none=True
)

x_logits = victim(
    x
)

x_loss = criterion(
    x_logits,
    unrelated_y,
)

x_grads = torch.autograd.grad(
    x_loss,
    tuple(victim.parameters()),
    create_graph=True,
    retain_graph=True,
    allow_unused=False,
)

distance_sq = torch.zeros(
    (),
    device=DEVICE,
)

for target_gradient, current_gradient in zip(
    target_grads,
    x_grads,
):

    distance_sq = (
        distance_sq
        + (
            target_gradient
            - current_gradient
        ).pow(2).sum()
    )

distance = torch.sqrt(
    distance_sq + 1e-12
)

input_gradient = torch.autograd.grad(
    distance,
    x,
    create_graph=False,
    retain_graph=False,
    allow_unused=False,
)[0]

print()
print(
    f"Gradient matching distance : "
    f"{distance.item():.10e}"
)

print(
    f"Input-gradient norm        : "
    f"{input_gradient.norm().item():.10e}"
)

print(
    f"Input-gradient min         : "
    f"{input_gradient.min().item():.10e}"
)

print(
    f"Input-gradient max         : "
    f"{input_gradient.max().item():.10e}"
)

print(
    f"Input-gradient nonzero     : "
    f"{input_gradient.ne(0).sum().item():,} / "
    f"{input_gradient.numel():,}"
)

assert distance.item() > 0

assert (
    input_gradient.norm().item() > 0
), (
    "CRITICAL FAILURE: gradient matching distance "
    "has no gradient with respect to the input image."
)

print()
print("✓ Gradient matching distance is differentiable")
print("✓ Nonzero gradient reaches the input image")
print()
print("✓ DIAGNOSTIC 3 COMPLETE")


GRADIENT DIAGNOSTIC 3 — SECOND-ORDER DIFFERENTIABILITY

Gradient matching distance : 3.5860828400e+01
Input-gradient norm        : 1.1679651260e+01
Input-gradient min         : -2.3206543922e-01
Input-gradient max         : 3.8990026712e-01
Input-gradient nonzero     : 196,608 / 196,608

✓ Gradient matching distance is differentiable
✓ Nonzero gradient reaches the input image

✓ DIAGNOSTIC 3 COMPLETE


### Results interpretation



## Pre-reconstruction diagnostic 4 — Gradient behavior on a genuine DPS diffusion sample

The previous checks use real CelebA images, whereas GGSS-R optimizes images produced by the FFHQ diffusion prior. This diagnostic generates an unconditional sample with the same DPS FFHQ model and DDIM machinery used by the attack, then measures its victim gradient. It tests whether the victim-gradient path remains numerically valid on images that actually lie on the diffusion sampling trajectory.


In [19]:
import yaml
from pathlib import Path

from guided_diffusion.unet import create_model
from guided_diffusion.gaussian_diffusion import create_sampler

print("=" * 70)
print("GRADIENT DIAGNOSTIC 4 — DPS FFHQ DIFFUSION SAMPLE")
print("=" * 70)

REPO_ROOT = Path(CODE_ROOT).resolve()

MODEL_CONFIG_PATH = (
    REPO_ROOT / "configs" / "model_config.yaml"
)

DIFFUSION_CONFIG_PATH = (
    REPO_ROOT / "configs" / "diffusion_ddim1000_config.yaml"
)

FFHQ_CHECKPOINT = (
    REPO_ROOT / "models" / "ffhq_10m.pt"
)

assert MODEL_CONFIG_PATH.is_file()
assert DIFFUSION_CONFIG_PATH.is_file()
assert FFHQ_CHECKPOINT.is_file()

print()
print("Repository :", REPO_ROOT)
print("Checkpoint :", FFHQ_CHECKPOINT)

with open(MODEL_CONFIG_PATH, "r") as f:
    model_config = yaml.load(
        f,
        Loader=yaml.FullLoader,
    )

with open(DIFFUSION_CONFIG_PATH, "r") as f:
    diffusion_config = yaml.load(
        f,
        Loader=yaml.FullLoader,
    )

print()
print("Image size :", model_config["image_size"])
print("Sampler    :", diffusion_config["sampler"])
print("Steps      :", diffusion_config["steps"])

diffusion_model = create_model(
    **model_config
)

diffusion_model = diffusion_model.to(
    DEVICE
)

diffusion_model.eval()

print()
print("✓ DPS/FFHQ diffusion model loaded")

sampler = create_sampler(
    **diffusion_config
)

print(
    "✓ DDIM sampler created"
)

print(
    "Sampler timesteps:",
    sampler.num_timesteps
)

torch.manual_seed(2026)

image_size = model_config["image_size"]

img = torch.randn(
    1,
    3,
    image_size,
    image_size,
    device=DEVICE,
)

print()
print("=" * 70)
print("GENERATING ORDINARY FFHQ SAMPLE")
print("=" * 70)

with torch.no_grad():

    for idx in range(
        sampler.num_timesteps - 1,
        -1,
        -1,
    ):

        time = torch.tensor(
            [idx],
            device=DEVICE,
        )

        out = sampler.p_sample(
            model=diffusion_model,
            x=img,
            t=time,
        )

        img = out["sample"].detach()

        if idx % 100 == 0 or idx == 0:
            print(
                f"timestep {idx:4d} | "
                f"min={img.min().item(): .4f} | "
                f"max={img.max().item(): .4f} | "
                f"mean={img.mean().item(): .4f}"
            )

diffusion_x = img.detach().clone()

print()
print("✓ Ordinary DPS FFHQ sample generated")
print(
    "Shape:",
    tuple(diffusion_x.shape)
)

from PIL import Image
import numpy as np

save_path = (
    (DATA_ROOT / "samples" / "diagnostic_dps_ffhq_sample.png")
)

display_x = (
    diffusion_x
    .squeeze(0)
    .cpu()
    .clamp(-1, 1)
    .add(1)
    .div(2)
    .permute(1, 2, 0)
    .numpy()
)

display_x = (
    display_x * 255
).round().astype(np.uint8)

Image.fromarray(
    display_x
).save(save_path)

print(
    "Saved sample:",
    save_path
)

print()
print("=" * 70)
print("TESTING DPS IMAGE THROUGH VICTIM")
print("=" * 70)

diffusion_x.requires_grad_(True)

victim.zero_grad(
    set_to_none=True
)

diffusion_logits = victim(
    diffusion_x
)

diffusion_y = torch.tensor(
    [0],
    dtype=torch.long,
    device=DEVICE,
)

diffusion_loss = criterion(
    diffusion_logits,
    diffusion_y,
)

diffusion_grads = torch.autograd.grad(
    diffusion_loss,
    tuple(victim.parameters()),
    create_graph=True,
    retain_graph=True,
    allow_unused=False,
)

gradient_sq_norm = torch.zeros(
    (),
    device=DEVICE,
)

gradient_nonzero = 0
gradient_elements = 0

for g in diffusion_grads:

    gradient_sq_norm = (
        gradient_sq_norm
        + g.pow(2).sum()
    )

    gradient_nonzero += (
        g.detach()
        .ne(0)
        .sum()
        .item()
    )

    gradient_elements += g.numel()

diffusion_gradient_norm = torch.sqrt(
    gradient_sq_norm
)

distance_sq = torch.zeros(
    (),
    device=DEVICE,
)

for target_g, diffusion_g in zip(
    target_grads,
    diffusion_grads,
):

    distance_sq = (
        distance_sq
        + (
            target_g
            - diffusion_g
        ).pow(2).sum()
    )

distance = torch.sqrt(
    distance_sq + 1e-12
)

input_gradient = torch.autograd.grad(
    distance,
    diffusion_x,
    create_graph=False,
    retain_graph=False,
    allow_unused=False,
)[0]

print()
print("=" * 70)
print("DPS DIFFUSION GRADIENT DIAGNOSTIC")
print("=" * 70)

print()
print(
    "Diffusion logits:"
)

print(
    diffusion_logits.detach().cpu()
)

print()
print(
    f"Diffusion-image loss       : "
    f"{diffusion_loss.item():.10e}"
)

print(
    f"Diffusion gradient norm    : "
    f"{diffusion_gradient_norm.item():.10e}"
)

print(
    f"Gradient nonzero           : "
    f"{gradient_nonzero:,} / "
    f"{gradient_elements:,}"
)

print(
    f"Target-vs-diffusion distance: "
    f"{distance.item():.10e}"
)

print(
    f"Input-gradient norm        : "
    f"{input_gradient.norm().item():.10e}"
)

print(
    f"Input-gradient min         : "
    f"{input_gradient.min().item():.10e}"
)

print(
    f"Input-gradient max         : "
    f"{input_gradient.max().item():.10e}"
)

print(
    f"Input-gradient nonzero     : "
    f"{input_gradient.ne(0).sum().item():,} / "
    f"{input_gradient.numel():,}"
)

assert diffusion_gradient_norm.item() > 0, (
    "FAIL: DPS-generated image produced a zero "
    "victim gradient."
)

assert distance.item() > 0, (
    "FAIL: DPS gradient is identical to target gradient."
)

assert input_gradient.norm().item() > 0, (
    "FAIL: gradient-distance objective does not "
    "propagate back to DPS-generated image."
)

print()
print("✓ DPS image produces a nonzero victim gradient")
print("✓ DPS gradient differs from target gradient")
print("✓ Gradient-distance objective reaches DPS image")
print()
print("✓ DIAGNOSTIC 4 COMPLETE")


GRADIENT DIAGNOSTIC 4 — DPS FFHQ DIFFUSION SAMPLE

Repository : /content/GGSS_R_unperturbed_reproduction/repo
Checkpoint : /content/GGSS_R_unperturbed_reproduction/repo/models/ffhq_10m.pt

Image size : 256
Sampler    : ddim
Steps      : 1000

✓ DPS/FFHQ diffusion model loaded
eta: 1
✓ DDIM sampler created
Sampler timesteps: 1000

GENERATING ORDINARY FFHQ SAMPLE
timestep  900 | min=-4.7188 | max= 4.5366 | mean=-0.0019
timestep  800 | min=-4.6289 | max= 4.7393 | mean=-0.0041
timestep  700 | min=-5.2298 | max= 4.3850 | mean=-0.0093
timestep  600 | min=-4.5099 | max= 4.5104 | mean=-0.0167
timestep  500 | min=-4.3092 | max= 4.4526 | mean=-0.0304
timestep  400 | min=-4.0297 | max= 4.3315 | mean=-0.0522
timestep  300 | min=-3.6524 | max= 3.4641 | mean=-0.0728
timestep  200 | min=-2.9906 | max= 3.2830 | mean=-0.0950
timestep  100 | min=-2.0958 | max= 2.0399 | mean=-0.1125
timestep    0 | min=-1.0000 | max= 1.0000 | mean=-0.1193

✓ Ordinary DPS FFHQ sample generated
Shape: (1, 3, 256, 256)
Save

### Results interpretation



## Pre-reconstruction diagnostic 5 — Original GGSS-R measurement operator

The preceding diagnostics reconstruct the relevant computations explicitly. This diagnostic closes the gap to the attack implementation by passing the diffusion sample through the authors’ original `ReconstructionOperator.forward()` path. Agreement with the controlled calculations confirms that the measurement operator used by the runner is producing the expected victim-gradient representation.


In [20]:
import os
import sys
from pathlib import Path

import torch
from PIL import Image
from torchvision import transforms

print("=" * 70)
print("GRADIENT DIAGNOSTIC 5 — ORIGINAL GGSS-R MEASUREMENT")
print("=" * 70)

assert "diffusion_x" in globals(), (
    "Diagnostic 4 has not been run in this notebook."
)

assert torch.is_tensor(diffusion_x)

print()
print("DPS image shape:", tuple(diffusion_x.shape))

REPO_ROOT = Path(CODE_ROOT).resolve()

MEASUREMENTS_PATH = (
    REPO_ROOT / "guided_diffusion" / "measurements.py"
)

MODEL_STATE_PATH = (
    REPO_ROOT / "model_state" / "MLP_1.pth"
)

assert MEASUREMENTS_PATH.is_file(), (
    f"Missing measurements.py:\n{MEASUREMENTS_PATH}"
)

assert MODEL_STATE_PATH.is_file(), (
    f"Missing repository checkpoint:\n{MODEL_STATE_PATH}"
)

print()
print("Original measurements.py:")
print(" ", MEASUREMENTS_PATH)

print()
print("Repository victim checkpoint:")
print(" ", MODEL_STATE_PATH)

if "VICTIM_CHECKPOINT" in globals():

    notebook_checkpoint = Path(
        VICTIM_CHECKPOINT
    ).resolve()

    if notebook_checkpoint.is_file():

        notebook_size = (
            notebook_checkpoint.stat().st_size
        )

        repo_size = (
            MODEL_STATE_PATH.stat().st_size
        )

        print()
        print("Notebook checkpoint:")
        print(" ", notebook_checkpoint)

        print()
        print(
            f"Notebook checkpoint size: "
            f"{notebook_size:,} bytes"
        )

        print(
            f"Repository checkpoint size: "
            f"{repo_size:,} bytes"
        )

        assert notebook_size == repo_size, (
            "Repository MLP_1.pth and notebook "
            "victim checkpoint have different sizes."
        )

        print()
        print(
            "✓ Repository checkpoint matches "
            "notebook checkpoint"
        )

OLD_CWD = Path.cwd()

os.chdir(REPO_ROOT)

print()
print("Temporary working directory:")
print(" ", Path.cwd())

if "guided_diffusion.measurements" in sys.modules:
    del sys.modules["guided_diffusion.measurements"]

from guided_diffusion.measurements import (
    ReconstructionOperator
)

print()
print(
    "✓ Original ReconstructionOperator imported"
)

operator = ReconstructionOperator(
    device=DEVICE
)

print()
print(
    "✓ Original GGSS-R ReconstructionOperator instantiated"
)

operator_input = (
    diffusion_x
    .detach()
    .clone()
    .to(DEVICE)
)

operator_input.requires_grad_(True)

print()
print("=" * 70)
print("RUNNING ORIGINAL ReconstructionOperator.forward()")
print("=" * 70)

operator_gradient = operator.forward(
    operator_input
)

operator_gradient_norm = (
    operator_gradient.norm()
)

operator_gradient_nonzero = (
    operator_gradient
    .detach()
    .ne(0)
    .sum()
    .item()
)

print()
print(
    "Operator gradient shape:"
)

print(
    " ",
    tuple(operator_gradient.shape)
)

print(
    "Operator gradient dtype:"
)

print(
    " ",
    operator_gradient.dtype
)

print()
print(
    f"Operator gradient norm: "
    f"{operator_gradient_norm.item():.10e}"
)

print(
    f"Operator gradient nonzero: "
    f"{operator_gradient_nonzero:,} / "
    f"{operator_gradient.numel():,}"
)

TARGET_IMAGE_PATH = TARGET_PATH

assert TARGET_IMAGE_PATH.is_file(), (
    f"Target image not found:\n{TARGET_IMAGE_PATH}"
)

target_pil = Image.open(
    TARGET_IMAGE_PATH
).convert("RGB")

target_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5),
    ),
])

target_operator_input = (
    target_transform(target_pil)
    .unsqueeze(0)
    .to(DEVICE)
)

target_operator_input.requires_grad_(True)

operator_target_gradient = operator.forward(
    target_operator_input
)

operator_target_gradient_norm = (
    operator_target_gradient.norm()
)

print()
print(
    "Target operator gradient norm:"
)

print(
    f"  "
    f"{operator_target_gradient_norm.item():.10e}"
)

operator_distance = torch.linalg.norm(
    operator_target_gradient
    - operator_gradient
)

print()
print("=" * 70)
print("ORIGINAL GGSS-R OPERATOR DISTANCE")
print("=" * 70)

operator_distance_value = (
    operator_distance.item()
)

print()
print(
    "Target-vs-DPS distance:"
)

print(
    f"  {operator_distance_value:.10e}"
)

operator_input_gradient = torch.autograd.grad(
    outputs=operator_distance,
    inputs=operator_input,
    create_graph=False,
    retain_graph=False,
    allow_unused=False,
)[0]

operator_input_gradient_norm = (
    operator_input_gradient.norm()
)

operator_input_gradient_min = (
    operator_input_gradient.min()
)

operator_input_gradient_max = (
    operator_input_gradient.max()
)

operator_input_gradient_nonzero = (
    operator_input_gradient
    .ne(0)
    .sum()
    .item()
)

print()
print(
    "Input-gradient norm:"
)

print(
    f"  {operator_input_gradient_norm.item():.10e}"
)

print(
    "Input-gradient min:"
)

print(
    f"  {operator_input_gradient_min.item():.10e}"
)

print(
    "Input-gradient max:"
)

print(
    f"  {operator_input_gradient_max.item():.10e}"
)

print(
    "Input-gradient nonzero:"
)

print(
    f"  {operator_input_gradient_nonzero:,}"
    f" / "
    f"{operator_input_gradient.numel():,}"
)

if "distance" in globals():

    diagnostic4_distance_value = (
        distance.item()
    )

    absolute_difference = abs(
        diagnostic4_distance_value
        - operator_distance_value
    )

    print()
    print("=" * 70)
    print("COMPARISON WITH DIAGNOSTIC 4")
    print("=" * 70)

    print()
    print(
        f"Diagnostic 4 distance : "
        f"{diagnostic4_distance_value:.10e}"
    )

    print(
        f"Diagnostic 5 distance : "
        f"{operator_distance_value:.10e}"
    )

    print(
        f"Absolute difference    : "
        f"{absolute_difference:.10e}"
    )

assert operator_gradient_norm.item() > 0, (
    "FAIL: original ReconstructionOperator produced "
    "a zero gradient for the DPS image."
)

assert operator_distance_value > 0, (
    "FAIL: original operator produced zero distance "
    "between target and DPS gradients."
)

assert operator_input_gradient_norm.item() > 0, (
    "FAIL: original operator's distance does not "
    "propagate back to the DPS image."
)

os.chdir(OLD_CWD)

print()
print("=" * 70)
print("✓ DIAGNOSTIC 5 COMPLETE")
print("=" * 70)

print()
print(
    "Original GGSS-R ReconstructionOperator:"
)

print(
    "  ✓ produced a nonzero victim gradient"
)

print(
    "  ✓ produced a nonzero target-vs-DPS distance"
)

print(
    "  ✓ propagated gradient back to the DPS image"
)

print()
print(
    "Working directory restored:"
)

print(
    " ",
    Path.cwd()
)


GRADIENT DIAGNOSTIC 5 — ORIGINAL GGSS-R MEASUREMENT

DPS image shape: (1, 3, 256, 256)

Original measurements.py:
  /content/GGSS_R_unperturbed_reproduction/repo/guided_diffusion/measurements.py

Repository victim checkpoint:
  /content/GGSS_R_unperturbed_reproduction/repo/model_state/MLP_1.pth

Temporary working directory:
  /content/GGSS_R_unperturbed_reproduction/repo

✓ Original ReconstructionOperator imported

✓ Original GGSS-R ReconstructionOperator instantiated

RUNNING ORIGINAL ReconstructionOperator.forward()

Operator gradient shape:
  (100729730,)
Operator gradient dtype:
  torch.float32

Operator gradient norm: 2.8391864300e+00
Operator gradient nonzero: 52,503,785 / 100,729,730

Target operator gradient norm:
  1.3358269930e+00

ORIGINAL GGSS-R OPERATOR DISTANCE

Target-vs-DPS distance:
  2.8006341457e+00

Input-gradient norm:
  1.3525224924e+00
Input-gradient min:
  -4.4072967023e-02
Input-gradient max:
  2.5951277465e-02
Input-gradient nonzero:
  196,608 / 196,608

COMPA

### Results interpretation



## 16. Run the unperturbed GGSS-R baseline

With the model, target, measurement path, and differentiability checks established, this cell executes the original 1000-step GGSS-R reconstruction with fixed guidance `m_r = 0.2` and no added gradient perturbation. Outputs are written to a dedicated baseline directory so the run can be inspected independently of the diagnostic setup.


In [21]:
import os
import sys
import subprocess
import shutil
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = PROJECT_ROOT
REPO_ROOT = (DRIVE_ROOT / "repo").resolve()
RESULTS_ROOT = (DRIVE_ROOT / "results").resolve()
BASELINE_ROOT = (RESULTS_ROOT / "baseline_fixed_mr_v3").resolve()

BASELINE_ROOT.mkdir(parents=True, exist_ok=True)
LOG_PATH = BASELINE_ROOT / f"run_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

DRIVE_TARGET = DRIVE_ROOT / "data" / "samples" / "celeba_target" / "target_0000.png"
REPO_TARGET_DIR = REPO_ROOT / "data" / "samples" / "celeba_target"
REPO_TARGET = REPO_TARGET_DIR / "target_0000.png"

assert DRIVE_TARGET.is_file(), f"Drive target missing: {DRIVE_TARGET}"

REPO_TARGET_DIR.mkdir(parents=True, exist_ok=True)
if (not REPO_TARGET.exists()) or (REPO_TARGET.stat().st_size != DRIVE_TARGET.stat().st_size):
    shutil.copy2(DRIVE_TARGET, REPO_TARGET)
    print("✓ Target image copied into repository data/samples/celeba_target/")
else:
    print("✓ Target image already present in repository")

required = [
    REPO_ROOT / "sample_condition_same_inputs.py",
    REPO_ROOT / "guided_diffusion" / "gaussian_diffusion.py",
    REPO_ROOT / "configs" / "model_config.yaml",
    REPO_ROOT / "configs" / "diffusion_ddim1000_config.yaml",
    REPO_ROOT / "configs" / "reconstruction_config.yaml",
    REPO_ROOT / "models" / "ffhq_10m.pt",
    REPO_ROOT / "model_state" / "MLP_1.pth",
    REPO_TARGET,
]

for p in required:
    assert p.exists(), f"Missing: {p}"

gd_text = (REPO_ROOT / "guided_diffusion" / "gaussian_diffusion.py").read_text()
assert "if record and (idx % 100 == 0 or idx == 0):" in gd_text
assert "os.makedirs(progress_dir, exist_ok=True)" in (
    REPO_ROOT / "sample_condition_same_inputs.py"
).read_text()

print("=" * 70)
print("GGSS-R BASELINE — PRE-RUN CHECK")
print("=" * 70)
print("Repository :", REPO_ROOT)
print("Output dir :", BASELINE_ROOT)
print("Log file   :", LOG_PATH)
print("Target     :", REPO_TARGET)
print()
print("✓ All required files present")
print()

cmd = [
    sys.executable, "sample_condition_same_inputs.py",
    "--model_config", "configs/model_config.yaml",
    "--diffusion_config", "configs/diffusion_ddim1000_config.yaml",
    "--task_config", "configs/reconstruction_config.yaml",
    "--gpu", "0",
    "--interval", "1",
    "--save_dir", str(BASELINE_ROOT),
    "--method", "GSS",
    "--data_root", "data/samples/celeba_target/",   # now exists inside the repo
    "--guidance_scale", "0.2",
]

print("Command:")
print(" ".join(cmd))
print()
print("=" * 70)
print("STARTING GGSS-R  (log is being written continuously to Drive)")
print("=" * 70)
print()

with open(LOG_PATH, "w", buffering=1) as log_file:
    process = subprocess.Popen(
        cmd,
        cwd=str(REPO_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    try:
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
            log_file.write(line)
    finally:
        process.stdout.close()

    return_code = process.wait()

print()
print("=" * 70)
if return_code == 0:
    print("✓ GGSS-R BASELINE COMPLETED")
    print("Results :", BASELINE_ROOT)
    print("Log     :", LOG_PATH)
else:
    print("✗ GGSS-R BASELINE FAILED")
    print(f"Return code: {return_code}")
    print("Full traceback is in the log file on Drive:")
    print(LOG_PATH)
    raise RuntimeError(f"GGSS-R exited with return code {return_code}")


✓ Target image copied into repository data/samples/celeba_target/
GGSS-R BASELINE — PRE-RUN CHECK
Repository : /content/GGSS_R_unperturbed_reproduction/repo
Output dir : /content/GGSS_R_unperturbed_reproduction/results/baseline_fixed_mr_v3
Log file   : /content/GGSS_R_unperturbed_reproduction/results/baseline_fixed_mr_v3/run_log_20260903_094458.txt
Target     : /content/GGSS_R_unperturbed_reproduction/repo/data/samples/celeba_target/target_0000.png

✓ All required files present

Command:
/usr/bin/python3 sample_condition_same_inputs.py --model_config configs/model_config.yaml --diffusion_config configs/diffusion_ddim1000_config.yaml --task_config configs/reconstruction_config.yaml --gpu 0 --interval 1 --save_dir /content/GGSS_R_unperturbed_reproduction/results/baseline_fixed_mr_v3 --method GSS --data_root data/samples/celeba_target/ --guidance_scale 0.2

STARTING GGSS-R  (log is being written continuously to Drive)

2026-09-03 09:45:06,422 [DPS] >> Device set to cuda:0.
09/03/2026 09:4

## Post-reconstruction diagnostic — Gradient identifiability of the recovered image

A successful optimization of gradient distance does not necessarily imply pixel-level recovery. This diagnostic compares the target, unrelated class-0 faces, saved predicted-clean-image snapshots, and the final reconstruction in both image space and victim-gradient space. It reports MSE, PSNR, LPIPS, gradient distance, relative gradient distance, and gradient cosine similarity to determine whether the final reconstruction is visually close to the target or merely produces a similar victim gradient.


In [22]:
import os
import sys
import gc
import math
import random
import subprocess
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import lpips

try:
    from datasets import load_dataset
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "datasets",
    ])
    from datasets import load_dataset

import numpy as np
import torch
import torch.nn as nn

from PIL import Image
from torchvision import transforms

DRIVE_ROOT = PROJECT_ROOT

REPO_ROOT = DRIVE_ROOT / "repo"

MODEL_PATH = (
    DRIVE_ROOT /
    "model_state" /
    "MLP_1.pth"
)

REPO_MODEL_PATH = (
    REPO_ROOT /
    "model_state" /
    "MLP_1.pth"
)

TARGET_PATH = (
    DRIVE_ROOT /
    "data" /
    "samples" /
    "celeba_target" /
    "target_0000.png"
)

TARGET_METADATA_PATH = (
    DRIVE_ROOT /
    "data" /
    "target_metadata.pt"
)

BASELINE_ROOT = (
    DRIVE_ROOT /
    "results" /
    "baseline_fixed_mr_v3" /
    "reconstruction"
)

assert REPO_ROOT.is_dir(), (
    f"Repository not found:\n{REPO_ROOT}"
)

assert MODEL_PATH.is_file(), (
    f"Victim checkpoint not found:\n{MODEL_PATH}"
)

assert TARGET_PATH.is_file(), (
    f"Target not found:\n{TARGET_PATH}"
)

assert TARGET_METADATA_PATH.is_file(), (
    f"Target metadata not found:\n{TARGET_METADATA_PATH}"
)

print("=" * 78)
print("GGSS-R VICTIM-GRADIENT DIAGNOSTIC")
print("=" * 78)

print()
print("Drive root       :", DRIVE_ROOT)
print("Repository       :", REPO_ROOT)
print("Victim checkpoint:", MODEL_PATH)
print("Target           :", TARGET_PATH)
print("Results          :", BASELINE_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from guided_diffusion.attacked_model import MLP_1

DEVICE = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print()
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))

metadata = torch.load(
    TARGET_METADATA_PATH,
    map_location="cpu",
)

target_index = int(
    metadata["target_index"]
)

target_label = int(
    metadata["target_label"]
)

assert target_label == 0

print()
print("=" * 78)
print("TARGET METADATA")
print("=" * 78)

print("Dataset      :", metadata["dataset_name"])
print("Split        :", metadata["dataset_split"])
print("Target index :", target_index)
print("Target label :", target_label)

CELEBA_TRANSFORM = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5),
    ),
])

model = MLP_1().to(DEVICE)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
)

model.load_state_dict(checkpoint)
model.eval()

lpips_model = lpips.LPIPS(net="alex").to(DEVICE)
lpips_model.eval()

print()
print("=" * 78)
print("VICTIM MODEL")
print("=" * 78)

print(model)

num_params = sum(
    p.numel()
    for p in model.parameters()
)

print()
print(f"Total parameters: {num_params:,}")

if REPO_MODEL_PATH.is_file():

    drive_bytes = MODEL_PATH.read_bytes()
    repo_bytes = REPO_MODEL_PATH.read_bytes()

    print()
    print(
        "Drive/repository checkpoint byte-identical:",
        drive_bytes == repo_bytes,
    )

else:

    print()
    print(
        "⚠ Repository copy of MLP_1.pth was not found."
    )

target_pil = Image.open(
    TARGET_PATH
).convert("RGB")

target_tensor = (
    CELEBA_TRANSFORM(target_pil)
    .unsqueeze(0)
    .to(DEVICE)
)

print()
print(
    "Target image size:",
    target_pil.size,
)

criterion = nn.CrossEntropyLoss()

attack_label_soft = torch.tensor(
    [[1.0, 0.0]],
    dtype=torch.float32,
    device=DEVICE,
)

def compute_gradient_info(x):
    """
    Compute the exact victim gradient used by GGSS-R.

    Returns:
        loss
        logits
        gradients: dict layer_name -> gradient tensor
    """

    x = x.detach().clone().requires_grad_(False)

    model.zero_grad(set_to_none=True)

    logits = model(x)

    loss = criterion(
        logits,
        attack_label_soft,
    )

    grads_tuple = torch.autograd.grad(
        outputs=loss,
        inputs=tuple(model.parameters()),
        create_graph=False,
        retain_graph=False,
    )

    gradients = {}

    for (name, _), grad in zip(
        model.named_parameters(),
        grads_tuple,
    ):
        gradients[name] = grad.detach()

    return (
        loss.detach(),
        logits.detach(),
        gradients,
    )

def gradient_total_norm(gradients):
    """
    L2 norm of the concatenation of all parameter gradients,
    without actually constructing the ~400 MB concatenated vector.
    """
    sq = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float64,
    )

    for g in gradients.values():
        sq += (
            g.detach()
            .double()
            .pow(2)
            .sum()
        )

    return torch.sqrt(sq).float()

def gradient_distance(g1, g2):
    """
    L2 distance between two full parameter-gradient vectors.
    """
    sq = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float64,
    )

    for name in g1:
        diff = (
            g1[name].double()
            - g2[name].double()
        )

        sq += diff.pow(2).sum()

    return torch.sqrt(sq).float()

def gradient_cosine_similarity(g1, g2):
    """
    Cosine similarity between two complete gradient vectors.
    """
    dot = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float64,
    )

    n1 = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float64,
    )

    n2 = torch.zeros(
        (),
        device=DEVICE,
        dtype=torch.float64,
    )

    for name in g1:

        a = g1[name].double()
        b = g2[name].double()

        dot += (a * b).sum()
        n1 += (a * a).sum()
        n2 += (b * b).sum()

    denom = torch.sqrt(n1 * n2)

    return (
        dot / denom
        if denom > 0
        else torch.tensor(
            float("nan"),
            device=DEVICE,
        )
    ).float()

print()
print("=" * 78)
print("COMPUTING TARGET GRADIENT")
print("=" * 78)

target_loss, target_logits, target_grads = (
    compute_gradient_info(target_tensor)
)

target_grad_norm = gradient_total_norm(
    target_grads
)

print()
print(
    "Target logits:",
    target_logits[0].detach().cpu().numpy()
)

print(
    f"Target loss: {target_loss.item():.10g}"
)

print(
    f"Total gradient norm: "
    f"{target_grad_norm.item():.10g}"
)

print()
print("=" * 78)
print("TARGET GRADIENT — LAYER-WISE DIAGNOSTICS")
print("=" * 78)

layer_norms = {}

for name, grad in target_grads.items():

    norm = torch.linalg.vector_norm(
        grad.float()
    )

    layer_norms[name] = norm

    print(
        f"{name:12s} : "
        f"{norm.item():.10g}"
    )

fc1_sq = (
    layer_norms["fc1.weight"] ** 2
    +
    layer_norms["fc1.bias"] ** 2
)

total_sq = target_grad_norm ** 2

fc1_fraction_energy = (
    fc1_sq / total_sq
)

fc1_fraction_norm = (
    torch.sqrt(fc1_sq) /
    target_grad_norm
)

print()
print(
    "First-layer gradient fraction "
    "(squared/L2 energy):",
    f"{fc1_fraction_energy.item():.8f}",
)

print(
    "First-layer gradient fraction "
    "(L2 norm):",
    f"{fc1_fraction_norm.item():.8f}",
)

grad_over_loss = (
    target_grad_norm /
    target_loss.abs().clamp_min(1e-30)
)

print()
print(
    "Gradient norm / |loss|:",
    f"{grad_over_loss.item():.10g}",
)

print()
print("=" * 78)
print("LOADING CELEBA TRAIN SPLIT FOR COMPARISON IMAGES")
print("=" * 78)

HF_CACHE = (PROJECT_ROOT / ".cache" / "huggingface")

HF_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("✓ Hugging Face authentication enabled from HF_TOKEN")
else:
    print("No HF_TOKEN environment variable found; continuing unauthenticated.")

celeba = load_dataset(
    "flwrlabs/celeba",
    split="train",
    cache_dir=str(HF_CACHE),
)

print()
print("CelebA train size:", len(celeba))
print("Columns:", celeba.column_names)

assert "Smiling" in celeba.column_names
assert "image" in celeba.column_names

smiling_labels = celeba["Smiling"]

class0_indices = [
    i
    for i, label in enumerate(smiling_labels)
    if int(label) == 0
]

class0_other = [
    i
    for i in class0_indices
    if i != target_index
]

print(
    "Class-0 training images:",
    len(class0_indices),
)

print(
    "Class-0 candidates after "
    "excluding target:",
    len(class0_other),
)

selection_rng = random.Random(20260814)

candidate_indices = selection_rng.sample(
    class0_other,
    min(12, len(class0_other)),
)

target_rgb_256 = np.asarray(
    target_pil.resize((256, 256)),
    dtype=np.float32,
) / 255.0

candidate_records = []

for idx in candidate_indices:

    item = celeba[idx]

    pil = item["image"].convert("RGB")
    pil_256 = pil.resize((256, 256))

    rgb = np.asarray(
        pil_256,
        dtype=np.float32,
    ) / 255.0

    mse = float(
        np.mean(
            (rgb - target_rgb_256) ** 2
        )
    )

    candidate_records.append(
        (
            mse,
            idx,
            pil_256,
        )
    )

candidate_records.sort(
    key=lambda x: x[0],
    reverse=True,
)

comparison_records = candidate_records[:3]

print()
print("Selected comparison images:")

for rank, (mse, idx, _) in enumerate(
    comparison_records,
    start=1,
):

    print(
        f"  {rank}. CelebA index {idx}"
        f" — RGB MSE vs target = {mse:.6f}"
    )

image_records = []

image_records.append(
    {
        "name": "TARGET",
        "source": "Drive target",
        "pil": target_pil,
    }
)

for rank, (_, idx, pil) in enumerate(
    comparison_records,
    start=1,
):

    image_records.append(
        {
            "name": f"CLASS0_{rank}",
            "source": f"CelebA train index {idx}",
            "pil": pil,
        }
    )

def rgb_array(pil):
    return np.asarray(
        pil.convert("RGB").resize((256, 256)),
        dtype=np.float32,
    ) / 255.0

def pixel_mse(a, b):
    aa = rgb_array(a)
    bb = rgb_array(b)

    return float(
        np.mean((aa - bb) ** 2)
    )

def pixel_l2(a, b):
    aa = rgb_array(a)
    bb = rgb_array(b)

    return float(
        np.linalg.norm(
            (aa - bb).reshape(-1)
        )
    )

def psnr(a, b):
    """
    PSNR between two RGB images represented in [0, 1].
    """
    aa = torch.from_numpy(
        rgb_array(a)
    ).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    bb = torch.from_numpy(
        rgb_array(b)
    ).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    mse = torch.mean((aa - bb) ** 2)

    if mse.item() == 0:
        return float("inf")

    return float(
        10.0 * torch.log10(1.0 / mse)
    )

def lpips_distance(a, b):
    """
    LPIPS distance using the AlexNet backbone.

    LPIPS expects inputs in [-1, 1].
    """
    aa = torch.from_numpy(
        rgb_array(a)
    ).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    bb = torch.from_numpy(
        rgb_array(b)
    ).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    aa = aa * 2.0 - 1.0
    bb = bb * 2.0 - 1.0

    with torch.no_grad():
        value = lpips_model(
            aa,
            bb,
        )

    return float(value.item())

print()
print("=" * 78)
print("TARGET vs OTHER CLASS-0 IMAGES")
print("=" * 78)

print()
print(
    f"{'Image':<12}"
    f"{'Pixel MSE':>14}"
    f"{'PSNR':>12}"
    f"{'LPIPS':>12}"
    f"{'Grad dist':>18}"
    f"{'Rel grad dist':>18}"
    f"{'Cosine':>14}"
)

print("-" * 78)

for record in image_records[1:]:

    x = (
        CELEBA_TRANSFORM(record["pil"])
        .unsqueeze(0)
        .to(DEVICE)
    )

    loss_i, logits_i, grads_i = (
        compute_gradient_info(x)
    )

    gd = gradient_distance(
        target_grads,
        grads_i,
    )

    rel_gd = (
        gd /
        target_grad_norm.clamp_min(1e-30)
    )

    cos = gradient_cosine_similarity(
        target_grads,
        grads_i,
    )

    pmse = pixel_mse(
        target_pil,
        record["pil"],
    )

    pps = psnr(
        target_pil,
        record["pil"],
    )

    plpips = lpips_distance(
        target_pil,
        record["pil"],
    )

    print(
        f"{record['name']:<12}"
        f"{pmse:>14.8f}"
        f"{pps:>12.4f}"
        f"{plpips:>12.6f}"
        f"{gd.item():>18.8g}"
        f"{rel_gd.item():>18.8g}"
        f"{cos.item():>14.8f}"
    )

    del x, loss_i, logits_i, grads_i

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print()
print("=" * 78)
print("LOCATING SAVED GGSS-R RECONSTRUCTIONS")
print("=" * 78)

x0_dir = (
    BASELINE_ROOT /
    "progresss" /
    "00000" /
    "x0"
)

recon_path = (
    BASELINE_ROOT /
    "recon" /
    "00000.png"
)

print()
print("x0 directory:", x0_dir)
print("Final recon :", recon_path)

assert x0_dir.is_dir(), (
    f"x0 directory not found:\n{x0_dir}"
)

x0_paths = sorted(
    x0_dir.glob("x_*.png"),
    reverse=True,
)

print()
print(
    "Found",
    len(x0_paths),
    "saved x0 images."
)

for p in sorted(x0_paths):
    print(" ", p.name)

print()
print("=" * 78)
print("SAVED x0 — PIXEL + VICTIM-GRADIENT DIAGNOSTIC")
print("=" * 78)

print()
print(
    f"{'x0':<12}"
    f"{'Pixel MSE':>14}"
    f"{'PSNR':>12}"
    f"{'LPIPS':>12}"
    f"{'Grad dist':>18}"
    f"{'Rel grad dist':>18}"
    f"{'Cosine':>14}"
)

print("-" * 78)

x0_results = []

for path in sorted(
    x0_paths,
    key=lambda p: int(
        p.stem.split("_")[1]
    ),
    reverse=True,
):

    pil = Image.open(
        path
    ).convert("RGB")

    x = (
        CELEBA_TRANSFORM(pil)
        .unsqueeze(0)
        .to(DEVICE)
    )

    loss_i, logits_i, grads_i = (
        compute_gradient_info(x)
    )

    gd = gradient_distance(
        target_grads,
        grads_i,
    )

    rel_gd = (
        gd /
        target_grad_norm.clamp_min(1e-30)
    )

    cos = gradient_cosine_similarity(
        target_grads,
        grads_i,
    )

    pmse = pixel_mse(
        target_pil,
        pil,
    )

    pps = psnr(
        target_pil,
        pil,
    )

    plpips = lpips_distance(
        target_pil,
        pil,
    )

    timestep = int(
        path.stem.split("_")[1]
    )

    x0_results.append(
        {
            "timestep": timestep,
            "pixel_mse": pmse,
            "gradient_distance": gd.item(),
            "relative_gradient_distance": rel_gd.item(),
            "gradient_cosine": cos.item(),
        }
    )

    print(
        f"x_{timestep:04d}"
        f"{pmse:>14.8f}"
        f"{pps:>12.4f}"
        f"{plpips:>12.6f}"
        f"{gd.item():>18.8g}"
        f"{rel_gd.item():>18.8g}"
        f"{cos.item():>14.8f}"
    )

    del x, loss_i, logits_i, grads_i, pil

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

if recon_path.is_file():

    print()
    print("=" * 78)
    print("FINAL RECONSTRUCTION PNG")
    print("=" * 78)

    recon_pil = Image.open(
        recon_path
    ).convert("RGB")

    x = (
        CELEBA_TRANSFORM(recon_pil)
        .unsqueeze(0)
        .to(DEVICE)
    )

    recon_loss, recon_logits, recon_grads = (
        compute_gradient_info(x)
    )

    recon_gd = gradient_distance(
        target_grads,
        recon_grads,
    )

    recon_rel_gd = (
        recon_gd /
        target_grad_norm.clamp_min(1e-30)
    )

    recon_cos = gradient_cosine_similarity(
        target_grads,
        recon_grads,
    )

    recon_pmse = pixel_mse(
        target_pil,
        recon_pil,
    )

    recon_psnr = psnr(
        target_pil,
        recon_pil,
    )

    recon_lpips = lpips_distance(
        target_pil,
        recon_pil,
    )

    recon_pl2 = pixel_l2(
        target_pil,
        recon_pil,
    )

    print()
    print(
        "Pixel MSE                 :",
        f"{recon_pmse:.10g}",
    )

    print(
        "PSNR                      :",
        f"{recon_psnr:.10g} dB",
    )

    print(
        "LPIPS                     :",
        f"{recon_lpips:.10g}",
    )

    print(
        "Pixel L2                  :",
        f"{recon_pl2:.10g}",
    )

    print(
        "Victim-gradient distance  :",
        f"{recon_gd.item():.10g}",
    )

    print(
        "Relative gradient distance:",
        f"{recon_rel_gd.item():.10g}",
    )

    print(
        "Gradient cosine similarity:",
        f"{recon_cos.item():.10g}",
    )

    print(
        "Reconstruction loss       :",
        f"{recon_loss.item():.10g}",
    )

    del x, recon_loss, recon_logits, recon_grads, recon_pil

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

else:

    print()
    print(
        "⚠ Final recon/00000.png was not found."
    )

print()
print("=" * 78)
print("DIAGNOSTIC COMPLETE")
print("=" * 78)



GGSS-R VICTIM-GRADIENT DIAGNOSTIC

Drive root       : /content/GGSS_R_unperturbed_reproduction
Repository       : /content/GGSS_R_unperturbed_reproduction/repo
Victim checkpoint: /content/GGSS_R_unperturbed_reproduction/model_state/MLP_1.pth
Target           : /content/GGSS_R_unperturbed_reproduction/data/samples/celeba_target/target_0000.png
Results          : /content/GGSS_R_unperturbed_reproduction/results/baseline_fixed_mr_v3/reconstruction

Device: cuda:0
GPU   : Tesla T4

TARGET METADATA
Dataset      : flwrlabs/celeba
Split        : train
Target index : 29786
Target label : 0
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 178MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth

VICTIM MODEL
MLP_1(
  (fc1): Linear(in_features=196608, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=2, bias=True)
  (relu): ReLU()
)

Total parameters: 100,729,730

Drive/repository checkpoint byte-identical: True

Target image size: (178, 218)

COMPUTING TARGET GRADIENT

Target logits: [-0.10914069 -4.377861  ]
Target loss: 0.01390261017
Total gradient norm: 1.335826993

TARGET GRADIENT — LAYER-WISE DIAGNOSTICS
fc1.weight   : 0.1433238834
fc1.bias     : 0.0006158770411
fc2.weight   : 0.9478946328
fc2.bias     : 0.002956494922
fc3.weight   : 0.9300523996
fc3.bias     : 0.01952520199

First-layer gradient fraction (squared/L2 energy): 0.01151184
First-layer gradient fraction (L2 norm): 0.10729324

Gradient norm / |loss|: 96.08461761

LOADING CELEBA TRAIN SPLIT FOR COMPARISON IMAGES
No HF_TOKEN environment va

...

Generating train split:   0%|          | 0/162770 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/19867 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19962 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/19 [00:00<?, ?it/s]


CelebA train size: 162770
Columns: ['image', 'celeb_id', '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive', 'Bags_Under_Eyes', 'Bald', 'Bangs', 'Big_Lips', 'Big_Nose', 'Black_Hair', 'Blond_Hair', 'Blurry', 'Brown_Hair', 'Bushy_Eyebrows', 'Chubby', 'Double_Chin', 'Eyeglasses', 'Goatee', 'Gray_Hair', 'Heavy_Makeup', 'High_Cheekbones', 'Male', 'Mouth_Slightly_Open', 'Mustache', 'Narrow_Eyes', 'No_Beard', 'Oval_Face', 'Pale_Skin', 'Pointy_Nose', 'Receding_Hairline', 'Rosy_Cheeks', 'Sideburns', 'Smiling', 'Straight_Hair', 'Wavy_Hair', 'Wearing_Earrings', 'Wearing_Hat', 'Wearing_Lipstick', 'Wearing_Necklace', 'Wearing_Necktie', 'Young']
Class-0 training images: 84690
Class-0 candidates after excluding target: 84689

Selected comparison images:
  1. CelebA index 14403 — RGB MSE vs target = 0.244126
  2. CelebA index 85760 — RGB MSE vs target = 0.237818
  3. CelebA index 128873 — RGB MSE vs target = 0.227590

TARGET vs OTHER CLASS-0 IMAGES

Image            Pixel MSE        PSNR       LPIPS

### Results interpretation



## Post-reconstruction diagnostic — Per-layer contribution to target–reconstruction gradient distance

A layer can have a large target-gradient norm without necessarily dominating the difference between the target and reconstructed gradients. To measure the quantity that is directly relevant to GGSS-R’s gradient-matching objective, this diagnostic recomputes both parameter-gradient vectors and decomposes their squared Euclidean distance by victim-model parameter tensor. For each layer it reports the absolute squared-distance contribution and its percentage of the total squared gradient distance. This reveals which parts of the victim model actually account for the remaining mismatch after reconstruction.


In [23]:
import torch
from PIL import Image

victim.eval()
recon_path = BASELINE_ROOT / "recon" / "00000.png"
assert recon_path.is_file(), f"Final reconstruction not found: {recon_path}"

criterion = torch.nn.CrossEntropyLoss()
label = torch.tensor([0], dtype=torch.long, device=DEVICE)

def parameter_gradients(image_path):
    pil = Image.open(image_path).convert("RGB")
    x = attack_transform(pil).unsqueeze(0).to(DEVICE)
    victim.zero_grad(set_to_none=True)
    logits = victim(x)
    loss = criterion(logits, label)
    grads = torch.autograd.grad(loss, tuple(victim.parameters()), create_graph=False, retain_graph=False)
    return [g.detach() for g in grads]

target_layer_grads = parameter_gradients(TARGET_PATH)
recon_layer_grads = parameter_gradients(recon_path)

rows = []
total_sq = sum((gt - gr).pow(2).sum() for gt, gr in zip(target_layer_grads, recon_layer_grads))
for (name, _), gt, gr in zip(victim.named_parameters(), target_layer_grads, recon_layer_grads):
    sq = (gt - gr).pow(2).sum()
    rows.append((name, float(sq.item()), float((100.0 * sq / total_sq.clamp_min(1e-30)).item())))

rows.sort(key=lambda x: x[1], reverse=True)
print(f"{'Parameter':<36}{'Squared distance':>20}{'Contribution (%)':>20}")
print("-" * 76)
for name, sq, pct in rows:
    print(f"{name:<36}{sq:>20.10e}{pct:>20.6f}")
print("-" * 76)
print(f"{'TOTAL':<36}{float(total_sq.item()):>20.10e}{100.0:>20.6f}")


Parameter                               Squared distance    Contribution (%)
----------------------------------------------------------------------------
fc2.weight                              1.3742947578e-01           74.985458
fc1.weight                              4.0072727948e-02           21.864828
fc3.weight                              5.7700332254e-03            3.148295
fc3.bias                                1.3975392221e-06            0.000763
fc2.bias                                1.0807392528e-06            0.000590
fc1.bias                                1.1827052049e-07            0.000065
----------------------------------------------------------------------------
TOTAL                                   1.8327483535e-01          100.000000


### Results interpretation

